# Поиск контекста для LLM

Соберём простой retrieval-baseline: преобразуем документы в TF-IDF-векторы и найдём наиболее релевантный контекст по cosine similarity.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
documents = pd.DataFrame([
    ("ml-pipeline", "ML pipeline", "Pipeline объединяет подготовку признаков и модель, снижая риск утечки данных."),
    ("cross-validation", "Cross-validation", "Кросс-валидация оценивает устойчивость модели на нескольких разбиениях выборки."),
    ("classification-metrics", "Classification metrics", "Precision, recall, F1 и ROC-AUC описывают разные стороны качества классификатора."),
    ("embeddings", "Text embeddings", "Текстовые embeddings представляют смысл фрагмента числовым вектором."),
    ("rag", "Retrieval augmented generation", "RAG сначала находит релевантный контекст, затем передаёт его языковой модели."),
    ("vector-database", "Vector database", "Векторная база хранит embeddings и выполняет быстрый поиск ближайших документов."),
    ("prompting", "Prompt design", "Хороший prompt задаёт роль, контекст, ограничения и формат ожидаемого ответа."),
    ("monitoring", "Model monitoring", "Мониторинг отслеживает качество, задержку, ошибки и изменение входных данных."),
], columns=["id", "title", "text"])
documents

,id,title,text
0,ml-pipeline,ML pipeline,Pipeline объединяет подготовку признаков и мод...
1,cross-validation,Cross-validation,Кросс-валидация оценивает устойчивость модели ...
2,classification-metrics,Classification metrics,"Precision, recall, F1 и ROC-AUC описывают разн..."
3,embeddings,Text embeddings,Текстовые embeddings представляют смысл фрагме...
4,rag,Retrieval augmented generation,"RAG сначала находит релевантный контекст, зате..."
5,vector-database,Vector database,Векторная база хранит embeddings и выполняет б...
6,prompting,Prompt design,"Хороший prompt задаёт роль, контекст, ограниче..."
7,monitoring,Model monitoring,"Мониторинг отслеживает качество, задержку, оши..."


In [3]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), lowercase=True)
document_matrix = vectorizer.fit_transform(documents["text"])

def search(query, limit=3):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_matrix).ravel()
    order = scores.argsort()[::-1][:limit]
    result = documents.iloc[order][["id", "title"]].copy()
    result["score"] = scores[order]
    return result.reset_index(drop=True)

In [4]:
search("как найти контекст для ответа языковой модели")

,id,title,score
0,rag,Retrieval augmented generation,0.378146
1,prompting,Prompt design,0.198475
2,cross-validation,Cross-validation,0.081888
